In [1]:

# ============================================================
# Shared setup (anonymized; paths are never printed)
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr

# Grayscale, publication style; no colour, no titles/captions in saved images.
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"image.cmap": "gray", "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 11, "savefig.dpi": 600})

DATA = Path("../data"); FIG = Path("../results/figures"); TAB = Path("../results/tables")
RNG = np.random.default_rng(42)

# Dataset registry (public sources) and decision domain.
DATASETS = {"telco": "telco_churn.csv", "adult": "adult.parquet",
            "credit_risk": "credit_risk.csv", "gmsc": "gmsc.parquet",
            "taiwan": "taiwan.parquet"}
DOMAIN = {"telco": "non-credit", "adult": "non-credit",
          "credit_risk": "consumer credit", "gmsc": "consumer credit",
          "taiwan": "consumer credit"}
# Grayscale shade + marker per domain (colour-free, print-safe).
STYLE = {"consumer credit": ("0.15", "o"), "corporate credit": ("0.0", "*"),
         "non-credit": ("0.45", "s")}
# Paper-1 anchor (Polish corporate bankruptcy; values reported in the first paper).
POLISH = {"gini": 2*0.778-1, "pbar": 0.0482, "naive_gap": 2.72, "indep_gap": 0.33}

def load(name):
    f = DATA / DATASETS[name]
    return pd.read_parquet(f) if str(f).endswith(".parquet") else pd.read_csv(f)

def prep(name, df):
    # Return numeric design matrix, binary outcome, and two pd-independent drivers (b1,b2).
    df = df.copy()
    if name == "telco":
        y = (df["Churn"].astype(str).str.strip() == "Yes").astype(int).values
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        X = df.drop(columns=["customerID", "Churn"]); b2col = "TotalCharges"
    elif name == "adult":
        y = df["HIGH_INCOME"].astype(int).values
        drop = ["income", "HIGH_INCOME", "AGE_YEARS", "SEX_STD", "EDU_STD", "VIP", "fnlwgt"]
        X = df.drop(columns=[c for c in drop if c in df]); b2col = "hours-per-week"
    elif name == "credit_risk":
        y = df["loan_status"].astype(int).values
        X = df.drop(columns=["loan_status"]); b2col = "person_income"
    elif name == "gmsc":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP"] if c in df]); b2col = "INCOME"
    elif name == "taiwan":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP", "AGE_YEARS", "SEX_STD"] if c in df]); b2col = "LIMIT_BAL"
    b2 = pd.to_numeric(df[b2col], errors="coerce").values if b2col in df else None
    Xn = pd.get_dummies(X, dummy_na=False).apply(pd.to_numeric, errors="coerce")
    return Xn, y, b2

def system_pd(X, y):
    # L2 logistic system model; out-of-fold PD via 5-fold stratified CV; in-fold median imputation.
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    pdh = np.zeros(len(y)); skf = StratifiedKFold(5, shuffle=True, random_state=42)
    for tr, te in skf.split(X, y):
        Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
        med = Xtr.median(numeric_only=True); Xtr = Xtr.fillna(med); Xte = Xte.fillna(med)
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
        clf.fit(sc.transform(Xtr), y[tr]); pdh[te] = clf.predict_proba(sc.transform(Xte))[:, 1]
    return pdh, roc_auc_score(y, pdh)

def pick_b1(X, pdh):
    # Near-orthogonal driver: numeric feature with the lowest |Spearman| vs PD (Attr43 analogue).
    best, br = None, 2.0
    for c in X.columns:
        v = X[c].values.astype(float); m = ~np.isnan(v)
        if m.sum() < 100 or np.nanstd(v) == 0: continue
        r = abs(spearmanr(v[m], pdh[m]).statistic)
        if not np.isnan(r) and r < br: br, best = r, v
    return best

QGRID = [0, 0.10, 0.20, 0.35, 0.55, 0.75, 0.90, 1.0]
def assign_grades(pdh):
    qs = np.quantile(pdh, QGRID); qs[0] -= 1e-9; qs[-1] += 1e-9
    return np.digitize(pdh, qs[1:-1])
SCHED = np.array([0.10, 0.15, 0.20, 0.35, 0.50, 0.70, 0.80])  # ord 0 best .. ord 6 worst
OVR = 0.35

def rank01(v):
    return pd.Series(v).rank(pct=True).fillna(0.5).values

def expected_gap(pi, y):
    # Exact expected upgrade-minus-none default-rate gap (percentage points).
    return (np.sum(pi * y) / np.sum(pi) - y.mean()) * 100.0

def sim_once(pi_firm, grade, G, rng):
    # One generative override draw; returns signed grade_diff (negative = upgrade) and masks.
    N = len(grade)
    overridden = rng.random(N) < OVR
    up = overridden & (rng.random(N) < pi_firm)
    down = overridden & ~up
    notch = np.where(rng.random(N) < 0.70, 1, 2)
    gd = np.zeros(N, int); gd[up] = -notch[up]; gd[down] = notch[down]
    final = np.clip(grade + gd, 0, G - 1)
    return final - grade, overridden, up

def save_fig(fig, name):
    # Save grayscale figure as PNG and PDF at 600 dpi; no caption; no path printed.
    fig.savefig(FIG / (name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(FIG / (name + ".pdf"), bbox_inches="tight")
    plt.close(fig)


# 05_parameter_sensitivity.ipynb -- gap over an override-rate x upgrade-spread grid, per DGM
rates = [0.20, 0.30, 0.40, 0.50]; uppers = [0.50, 0.65, 0.80, 0.95]
recs = []
for name in DATASETS:
    sc = pd.read_parquet(TAB / ("scores_" + name + ".parquet"))
    y = sc["y"].values; grade = sc["grade"].values.astype(int)
    drivers = {"naive": None, "b1": rank01(sc["b1"].values), "b2": rank01(sc["b2"].values)}
    for dgm, r01 in drivers.items():
        for orr in rates:
            for up in uppers:
                if dgm == "naive":
                    sch = np.interp(np.linspace(0.071, 0.929, 7), [0.071, 0.929], [0.10, up])
                    pif = sch[grade]
                else:
                    pif = 0.10 + (up-0.10)*r01
                g = expected_gap(pif, y)  # override-rate cancels in the expected gap; kept for grid labelling
                recs.append({"dataset": name, "dgm": dgm, "rate": orr, "upper": up, "gap_pp": round(g, 3)})
grid = pd.DataFrame(recs); grid.to_csv(TAB / "sensitivity_grid.csv", index=False)
summ = grid.groupby("dgm")["gap_pp"].agg(["min", "max", "mean"]).round(3)
summ.to_csv(TAB / "sensitivity_summary.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, dgm in zip(axes, ["naive", "b1", "b2"]):
    piv = grid[grid.dgm == dgm].groupby(["rate", "upper"])["gap_pp"].mean().unstack()
    sns.heatmap(piv, ax=ax, cmap="Greys", annot=True, fmt=".1f", cbar=False,
                vmin=grid["gap_pp"].min(), vmax=grid["gap_pp"].max(), linewidths=0.5, linecolor="w")
    ax.set_xlabel("upgrade-prob upper (" + dgm + ")"); ax.set_ylabel("override rate" if dgm == "naive" else "")
save_fig(fig, "fig05_sensitivity")
summ


,min,max,mean
dgm,,,
b1,-0.377,0.100,-0.071
b2,-5.252,5.065,-1.656
naive,2.348,11.298,7.423
